# FASE 2. LIMPIEZA DE LOS DATOS - ESTACIONES (EEA)

## 1. Carga de datos originales

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("TFM_Estaciones_Valencia") \
    .getOrCreate()

# Ruta del archivo 
ruta_estaciones = "gs://tfm_unir1/estaciones/eea_metadatos.tsv"

# Cargar los datos
df_raw = (spark.read
                .option("header", True)
                .option("sep", ",")
                .option("quote", '"')
                .option("escape", '"')
                .option("multiline", True)
                .csv(ruta_estaciones))

print("Registros originales:", df_raw.count())
print("Columnas:", len(df_raw.columns))

26/10/24 19:05:10 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Registros originales: 187994
Columnas: 70


## 2. Selección de columnas de interés 

In [6]:
# Armamos dataframe con las columnas que usaremos filtrando solo las estaciones de Valencai 46250
df_estaciones = (df_raw
                .filter(F.col("Air Quality Station Nat Code").startswith("46250"))
                .select(
                    F.col("Air Quality Station Nat Code").alias("id_estacion"),
                    F.col("Air Quality Station EoI Code").alias("codigo_eea"),
                    F.col("Air Quality Station Name").alias("nombre_estacion"),
                    F.col("Air Quality Station Type").alias("tipo_estacion"),
                    F.col("Air Quality Station Area").alias("zona"),
                    F.col("Latitude").alias("latitud"),
                    F.col("Longitude").alias("longitud"),
                    F.col("Altitude").alias("altitud"),
                    F.col("Operational Activity Begin").alias("fecha_alta"),
                    F.col("Operational Activity End").alias("fecha_baja"),
                    F.col("Year").alias("anio"),
                    F.col("Air Pollutant").alias("contaminante")

                ))

df_estaciones.show(5)

print("Total de registros seleccionados:", df_estaciones.count())

+-----------+----------+--------------------+-------------+-----+---------+---------+---------+-------------------+----------+----+------------+
|id_estacion|codigo_eea|     nombre_estacion|tipo_estacion| zona|  latitud| longitud|  altitud|         fecha_alta|fecha_baja|anio|contaminante|
+-----------+----------+--------------------+-------------+-----+---------+---------+---------+-------------------+----------+----+------------+
|   46250030|   ES1239A|VALÈNCIA-PISTA DE...|      traffic|urban|39.456110|-0.375830|11.000000|01/07/1992 00:00:00|      null|2025|         SO2|
|   46250030|   ES1239A|VALÈNCIA-PISTA DE...|      traffic|urban|39.456110|-0.375830|11.000000|01/01/2009 00:00:00|      null|2025|        PM10|
|   46250030|   ES1239A|VALÈNCIA-PISTA DE...|      traffic|urban|39.456110|-0.375830|11.000000|01/01/2018 00:00:00|      null|2025|        PM10|
|   46250030|   ES1239A|VALÈNCIA-PISTA DE...|      traffic|urban|39.456110|-0.375830|11.000000|01/07/1992 00:00:00|      null|2025

Total de registros seleccionados: 390


## 3. Separación de los datos según su frecuencia temporal

En el caso de estaciones, esto no es necesario

## 4. Limpieza de los datos 

### Tratamiento de valores no válidos

In [7]:
import pandas as pd
from pyspark.sql import Window

# Conservamos solo los registros de 2025 (los registros anteriores se relacionan con estaciones dadas de baja/cerradas)
df_estaciones_clean = df_estaciones.filter (F.col("anio").isNotNull())

print("Total de registros seleccionados:", df_estaciones_clean.count())
print("Estaciones:", df_estaciones_clean.select("id_estacion").distinct().count())

Total de registros seleccionados: 107


Estaciones: 9


In [19]:
# Agrupando por estacion y obtener una fila por estacion

df_estaciones_clean = (df_estaciones_clean
    .groupBy("id_estacion", "codigo_eea", "nombre_estacion", "tipo_estacion", "zona")
    .agg(
        F.max("latitud").alias("latitud"),
        F.max("longitud").alias("longitud"),
        F.max("altitud").alias("altitud"),
        F.min(F.to_date("fecha_alta", "dd/MM/yyyy HH:mm:ss")).alias("fecha_alta"),
        F.max(F.to_date("fecha_baja", "dd/MM/yyyy HH:mm:ss")).alias("ultima_baja"),
        F.sum(F.col("fecha_baja").isNull().cast("int")).alias("sensores_activos")))

print("Total de registros seleccionados:", df_estaciones_clean.count())
df_estaciones_clean.orderBy("id_estacion").toPandas()


Total de registros seleccionados: 9


,id_estacion,codigo_eea,nombre_estacion,tipo_estacion,zona,latitud,longitud,altitud,fecha_alta,ultima_baja,sensores_activos
0,46250030,ES1239A,VALÈNCIA-PISTA DE SILLA,traffic,urban,39.456110,-0.375830,11.000000,1992-07-01,None,1
1,46250043,ES1619A,VALÈNCIA-VIVERS,background,urban,39.478060,-0.368330,11.000000,2002-01-01,None,1
2,46250046,ES1885A,VALÈNCIA-POLITÈCNIC,background,suburban,39.480300,-0.336400,7.000000,2008-04-04,None,1
3,46250047,ES1912A,VALÈNCIA - AVD. FRANCIA,traffic,urban,39.457500,-0.342780,7.000000,2009-01-01,None,1
4,46250048,ES1926A,VALÈNCIA - MOLÍ DEL SOL,traffic,suburban,39.481100,-0.408300,11.000000,2009-08-06,None,1
5,46250050,ES1970A,VALÈNCIA - BULEVARD SUD,traffic,urban,39.450280,-0.396390,15.000000,2010-05-24,None,1
6,46250051,ES2052A,VALENCIA-ALBUFERA,background,rural,39.315600,-0.318600,1.000000,2014-01-01,2014-10-31,0
7,46250054,ES2095A,VALÈNCIA-CENTRE,traffic,urban,39.448330,-0.376470,2.000000,2019-01-01,None,1
8,46250055,ES2163A,VALÈNCIA OLIVERETA,traffic,urban,39.469220,-0.406050,24.000000,2022-05-02,None,1


In [20]:
# Estado de la estacion. Si tiene al menos un sensor activo, queda sin fecha de baja
# Si esta cerrada, su fecha de baja es la del ultimo sensor que se dio de baja
df_estaciones_clean = (df_estaciones_clean \
    .withColumn("activa", F.col("sensores_activos") > 0) 
    .withColumn("fecha_baja", F.when(F.col("activa"), None).otherwise(F.col("ultima_baja")))
    .drop("ultima_baja", "sensores_activos"))
    

print("Total de registros seleccionados:", df_estaciones_clean.count())

Total de registros seleccionados: 9


### Estandarización de tipo de datos


In [22]:
# Convertimos latitud, longitud y altittud a numero decimal
for columna in ["latitud", "longitud", "altitud"]:
    df_estaciones_clean = df_estaciones_clean.withColumn(columna, F.col(columna).cast("double"))

df_estaciones_clean.printSchema()


root
 |-- id_estacion: string (nullable = true)
 |-- codigo_eea: string (nullable = true)
 |-- nombre_estacion: string (nullable = true)
 |-- tipo_estacion: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- latitud: double (nullable = true)
 |-- longitud: double (nullable = true)
 |-- altitud: double (nullable = true)
 |-- fecha_alta: date (nullable = true)
 |-- activa: boolean (nullable = true)
 |-- fecha_baja: date (nullable = true)



In [28]:
# Unificamos nombres de estaciones quitando el prefijo "VALENCIA" y guiones
df_estaciones_clean = df_estaciones_clean.withColumn(
    "nombre_estacion",
    F.trim(F.regexp_replace("nombre_estacion", "^VAL.{1,2}NCIA[ -]*", "")))

df_estaciones_clean.select ("id_estacion","nombre_estacion").orderBy("id_estacion").toPandas()

,id_estacion,nombre_estacion
0,46250030,PISTA DE SILLA
1,46250043,VIVERS
2,46250046,POLITÈCNIC
3,46250047,AVD. FRANCIA
4,46250048,MOLÍ DEL SOL
5,46250050,BULEVARD SUD
6,46250051,ALBUFERA
7,46250054,CENTRE
8,46250055,OLIVERETA


In [29]:
# Traducir zona y tipo de estacion al castellano
# Traducir tipo de estación y zona al castellano
df_estaciones_clean = (df_estaciones_clean
    .withColumn("tipo_estacion",
        F.when(F.col("tipo_estacion") == "traffic", "Tráfico")
         .when(F.col("tipo_estacion") == "background", "Fondo")
         .otherwise(F.col("tipo_estacion")))
    .withColumn("zona",
        F.when(F.col("zona") == "urban", "Urbana")
         .when(F.col("zona") == "suburban", "Suburbana")
         .when(F.col("zona") == "rural", "Rural")
         .otherwise(F.col("zona"))))

df_estaciones_clean.select("id_estacion", "nombre_estacion", "tipo_estacion", "zona") \
    .orderBy("id_estacion").toPandas()

,id_estacion,nombre_estacion,tipo_estacion,zona
0,46250030,PISTA DE SILLA,Tráfico,Urbana
1,46250043,VIVERS,Fondo,Urbana
2,46250046,POLITÈCNIC,Fondo,Suburbana
3,46250047,AVD. FRANCIA,Tráfico,Urbana
4,46250048,MOLÍ DEL SOL,Tráfico,Suburbana
5,46250050,BULEVARD SUD,Tráfico,Urbana
6,46250051,ALBUFERA,Fondo,Rural
7,46250054,CENTRE,Tráfico,Urbana
8,46250055,OLIVERETA,Tráfico,Urbana


### Verificación de trafico tras la limpieza

In [33]:
df_estaciones_clean.agg(
    F.count("*").alias("registros"),
    F.countDistinct("id_estacion").alias("estaciones_unicas"),
    F.sum(F.col("nombre_estacion").isNull().cast("int")).alias("nombre_nulos"),
    F.sum(F.col("tipo_estacion").isNull().cast("int")).alias("tipo_nulos"),
    F.sum(F.col("latitud").isNull().cast("int")).alias("latitud_nulos"),        
    F.sum(F.col("longitud").isNull().cast("int")).alias("longitud_nulos")
).show()

+---------+-----------------+------------+----------+-------------+--------------+
|registros|estaciones_unicas|nombre_nulos|tipo_nulos|latitud_nulos|longitud_nulos|
+---------+-----------------+------------+----------+-------------+--------------+
|        9|                9|           0|         0|            0|             0|
+---------+-----------------+------------+----------+-------------+--------------+



## 5. Validación final de los datos preparados

In [35]:
# Comparar los registros antes y después de la limpieza
# En el caso de estaciones tendremos diferencias porque hemos borrado y agrupado datos
datos_originales = df_estaciones.count()
datos_limpios = df_estaciones_clean.count()

print("Registros originales:", datos_originales)
print("Registros preparados:", datos_limpios)
print("Reducción:", datos_originales - datos_limpios,  f"({(1 - datos_limpios / datos_originales) * 100:.1f} %)")


Registros originales: 390
Registros preparados: 9
Reducción: 381 (97.7 %)


**Conclusiones:**
- La limpieza reduce el conjunto de 390 a 9 registros (−97,7 %). 
- Esto se debe a que agrupamos datos de las estaciones que aparecian multiplicados debido a los contaminantes. En nuestro caso, solo necesitamos el inventario de estaciones

## 8. Exportación de los datasets limpios

In [36]:
# Rutas de salida
ruta_estaciones = "gs://tfm_unir1/processed/estaciones/"

# Guardar datos diarios
df_estaciones_clean.write \
    .mode("overwrite") \
    .parquet(ruta_estaciones)

print("Datos limpios guardados correctamente")

Datos limpios guardados correctamente


In [39]:
# Verificacion final
df_silver = spark.read.parquet("gs://tfm_unir1/processed/estaciones/")

df_silver.printSchema()

df_silver.agg(
    F.count("*").alias("registros"),
    F.countDistinct("id_estacion").alias("estaciones"),
    F.countDistinct("nombre_estacion").alias("nombre_estaciones"),
    F.min("latitud").alias("latitud_minima"),
    F.max("latitud").alias("latitud_maxima"),
    F.min("longitud").alias("longitud_minima"),
    F.max("longitud").alias("longitud_maxima")
    
).show(truncate=False)


root
 |-- id_estacion: string (nullable = true)
 |-- codigo_eea: string (nullable = true)
 |-- nombre_estacion: string (nullable = true)
 |-- tipo_estacion: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- latitud: double (nullable = true)
 |-- longitud: double (nullable = true)
 |-- altitud: double (nullable = true)
 |-- fecha_alta: date (nullable = true)
 |-- activa: boolean (nullable = true)
 |-- fecha_baja: date (nullable = true)
 |-- nomre_estacion: string (nullable = true)

+---------+----------+-----------------+--------------+--------------+---------------+---------------+
|registros|estaciones|nombre_estaciones|latitud_minima|latitud_maxima|longitud_minima|longitud_maxima|
+---------+----------+-----------------+--------------+--------------+---------------+---------------+
|9        |9         |9                |39.3156       |39.4811       |-0.4083        |-0.3186        |
+---------+----------+-----------------+--------------+--------------+---------------+